In [2]:
import numpy as np
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp

from singleWaveUNET import singleWaveUNet

In [3]:
simsNP = np.load("data/sample_16_19_sims.npy")
maskNP = np.load("data/sample_16_19_mask.npy")
print(simsNP.shape)

(20000, 7, 3, 128, 128)


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

# ----------------------------
# Dataset
# ----------------------------
class SimulationDataset(Dataset):
    """
    Expects data of shape [T, 3, H, W] — a time series of simulation snapshots.
    Returns (input: [9, H, W], target: [3, H, W]) pairs.
    """
    def __init__(self, data: torch.Tensor, mask: torch.Tensor, mean=None, std=None):
        # data: [N, 7, 3, H, W]
        self.x = torch.cat([
            data[:, 0, :, :, :],   # [N, 3, H, W]
            data[:, 1, :, :, :],   # [N, 3, H, W]
            data[:, 2, :, :, :],   # [N, 3, H, W]
            data[:, 3, :, :, :],   # [N, 3, H, W]
        ], dim=1).float()                  # [N, 9, H, W]

        #self.y = (data[:, 4, :, :, :] - data[:, 3, :, :, :]).float()   # [N, 3, H, W]
        self.y = torch.cat([
            (data[:, 4, :, :, :] - data[:, 3, :, :, :]),   # [N, 3, H, W]
            (data[:, 5, :, :, :] - data[:, 4, :, :, :]),   # [N, 3, H, W]
            (data[:, 6, :, :, :] - data[:, 5, :, :, :]),   # [N, 3, H, W]
        ], dim=1).float() 

        self.mask = mask  # [N, H, W]

        # Compute stats from this data if not provided (i.e. training set)
        if mean is None:
            # mean/std over N, H, W — keep channel dim → shape [1, 9, 1, 1]
            self.mean = (
                data[:, 0].float().mean(dim=(0, 2, 3), keepdim=True) +
                data[:, 1].float().mean(dim=(0, 2, 3), keepdim=True) +
                data[:, 2].float().mean(dim=(0, 2, 3), keepdim=True) + 
                data[:, 3].float().mean(dim=(0, 2, 3), keepdim=True)
            ) / 4.0   
            # For std, pool all timesteps together for a more stable estimate
            self.std = torch.cat([
                data[:, 0].float(),
                data[:, 1].float(),
                data[:, 2].float(),
                data[:, 3].float(),                
            ], dim=0).std(dim=(0, 2, 3), keepdim=True)  # [1, 4, 1, 1]            

        else:
            self.mean = mean
            self.std  = std

        mean_x = self.mean.repeat(1, 4, 1, 1)  # [1, 9, 1, 1]
        std_x  = self.std.repeat(1, 4, 1, 1)   # [1, 9, 1, 1]

        std_y  = self.std.repeat(1, 3, 1, 1)   # [1, 9, 1, 1]
        # Normalize x
        self.x = (self.x - mean_x) / (std_x + 1e-8)        

        # Normalize y using only the first 3 channels stats (single timestep)
        self.y = self.y  / (std_y + 1e-8)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx], self.mask[idx]  # [9, H, W], [3, H, W], [H, W]


####################################################################
####################################################################
def masked_relative_l2_loss(pred, target, mask, eps=1e-8):
    """
    Relative/scaled L2 loss.

    Computes:

        ||pred - target||_2 / ||target||_2

    only on valid pixels where mask == 0.

    pred:   [B, C, H, W]
    target: [B, C, H, W]
    mask:   [B, H, W]
    """

    # valid pixels
    valid = (mask == 0).unsqueeze(1).expand_as(pred)

    # apply mask
    pred_valid = pred * valid
    target_valid = target * valid

    # flatten per sample
    pred_valid = pred_valid.reshape(pred.shape[0], -1)
    target_valid = target_valid.reshape(target.shape[0], -1)

    # numerator: ||pred-target||_2
    num = torch.norm(pred_valid - target_valid, p=2, dim=1)

    # denominator: ||target||_2
    den = torch.norm(target_valid, p=2, dim=1) + eps

    # relative error per sample
    loss = num / den

    return loss.mean()
####################################################################
def masked_loss(pred, target, mask):
    """Only compute loss on pixels where mask == 0."""
    # mask: [B, H, W] → expand to [B, 3, H, W]
    valid = (mask == 0).unsqueeze(1).expand_as(pred)
    # return nn.functional.mse_loss(pred[valid], target[valid])
    return nn.functional.l1_loss(pred[valid], target[valid])
####################################################################
def create_spatial_weight(h=128, w=128, sigma=32):

    y = torch.arange(h).float()
    x = torch.arange(w).float()

    yy, xx = torch.meshgrid(y, x, indexing='ij')

    cy = (h - 1) / 2
    cx = (w - 1) / 2

    dist2 = (yy - cy) ** 2 + (xx - cx) ** 2

    weight = torch.exp(-dist2 / (2 * sigma ** 2))

    # normalize so mean weight ≈ 1
    weight = weight / weight.mean()

    return weight
####################################################################
def weighted_charbonnier_loss(
    pred,
    target,
    mask,
    spatial_weight,
    eps=1e-3,
):

    valid = (mask == 0).unsqueeze(1)

    # [1,1,H,W]
    spatial_weight = spatial_weight.to(pred.device)
    spatial_weight = spatial_weight.unsqueeze(0).unsqueeze(0)

    diff = pred - target

    loss = torch.sqrt(diff * diff + eps * eps)

    # apply:
    # - validity mask
    # - spatial weighting
    weighted_loss = loss * valid * spatial_weight

    return weighted_loss.sum() / (valid * spatial_weight).sum()
####################################################################
def myLoss(    pred,
    target,
    mask,
    spatial_weight,
    eps=1e-3,):

    return masked_loss(pred, target, mask)
    #return weighted_charbonnier_loss(pred, target, mask, spatial_weight, eps)
####################################################################
####################################################################
####################################################################
# ----------------------------
# Training loop
# ----------------------------
def train(
    model: nn.Module,
    data: torch.Tensor,          # [T, 3, H, W]
    mask: torch.Tensor,          # [T, H, W]
    epochs: int = 100,
    batch_size: int = 8,
    lr: float = 1e-4,
    val_split: float = 0.2,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
):
    print(f"Using device: {device}")
    model = model.to(device)

    # if torch.cuda.device_count() > 1:
    #     print(f"Using {torch.cuda.device_count()} GPUs")
    #     model = nn.DataParallel(model)

    # Split into train/val
    # Split raw data first
    N = len(data)
    n_test  = int(N * 0.2)
    n_train = N - n_test

    train_data = data[:n_train]
    train_mask = mask[:n_train]
    val_data   = data[n_train:]
    val_mask   = mask[n_train:]
    train_size = len(train_data)
    val_size   = len(val_data)

    # Train set computes its own stats
    train_set = SimulationDataset(train_data,train_mask)

    # Val/test reuse train stats — critical to avoid data leakage
    val_set  = SimulationDataset(val_data,val_mask, mean=train_set.mean, std=train_set.std)

    # Save stats for inference later
    torch.save({"mean": train_set.mean, "std": train_set.std}, "norm_stats.pt")

    # dataset = SimulationDataset(data)
    # val_size = int(len(dataset) * val_split)
    # train_size = len(dataset) - val_size
    # train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False)

    spatial_weight = create_spatial_weight(h=128,w=128,sigma=96)
    spatial_weight1 = create_spatial_weight(h=128,w=128,sigma=64)
    spatial_weight2 = create_spatial_weight(h=128,w=128,sigma=32)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    #optimizer = torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")

    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        train_loss = 0.0
        for x, y,m in train_loader:
            x, y,m = x.to(device), y.to(device), m.to(device)
            optimizer.zero_grad()
            pred = model(x[:, -9:])
            
            # pred = 2.0*pred + x[:, -6:-3]
            
            #loss = criterion(pred, y)
            #loss = masked_loss(pred, y, m) 
            loss = myLoss(pred,y[:,:3],m,spatial_weight)
            
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x.size(0)
        train_loss /= train_size

        # --- Validate ---
        model.eval()
        val_loss,val_loss1, val_loss2, val_loss3 = 0.0, 0.0, 0.0,0.0

        with torch.no_grad():
            for x, y,m in val_loader:
                x, y,m = x.to(device), y.to(device), m.to(device)                
                ## next 4 days
                pred1 = model(x[:,-9:])
                val_loss1 += myLoss(pred1,y[:,:3],m,spatial_weight).item() * x.size(0)   
                ## next 8 days                
                x2 = torch.cat([x[:, 3:6],x[:, 6:9], pred1+x[:, -3:]], dim=1)
                # x2 = torch.cat([x[:, 3:6],x[:, 6:9], 2.0*pred1+x[:, -6:-3]], dim=1)
                pred2 = model(x2)
                val_loss2 += myLoss(pred2,y[:,3:6],m,spatial_weight1).item() * x.size(0)  
                ## next 12 days
                x3 = torch.cat([x[:, 6:9],x2[:,-3:],pred2+x2[:, -3:]], dim=1)
                # x3 = torch.cat([x[:, 6:9],x2[:,-3:],2.0*pred2+x2[:, -6:-3]], dim=1)
                pred3 = model(x3)
                val_loss3 += myLoss(pred3,y[:,6:],m,spatial_weight2).item() * x.size(0)  

                #val_loss += criterion(pred, y).item() * x.size(0)
                #val_loss += masked_loss(pred, y, m).item() * x.size(0)
                #val_loss += weighted_charbonnier_loss(pred,y,m,spatial_weight).item() * x.size(0)   
        val_loss1 /= val_size
        val_loss2 /= val_size
        val_loss3 /= val_size

        scheduler.step(val_loss1)  # Use the first validation loss for scheduling

        val_loss = val_loss1 + 0.1*val_loss2 + 0.1*val_loss3
        # --- Checkpoint ---
        if val_loss < best_val_loss:
            print(f"epoch {epoch} | val loss {val_loss:.6f} | > (improvement of {best_val_loss - val_loss:.6f})")
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_modelZ.pth")

        if epoch % 1 == 0 or epoch == 1:
            print(f"Epoch {epoch:04d} | Train Loss: {train_loss:.6f} | Val Loss:{val_loss:.6f} > {val_loss1:.6f}, {val_loss2:.6f}, {val_loss3:.6f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

    print(f"\nTraining complete. Best val loss: {best_val_loss:.6f}")
    return model


# ----------------------------
# Usage
# ----------------------------
# model = DSCMS(in_channels=9, out_channels=3)
# model = smp.Unet(
#     encoder_name="resnet18",
#     in_channels=9,
#     classes=3,
# )

model = singleWaveUNet(
    in_channels=9,
    out_channels=3,
    depth=3,                 # variable depth
    base_channels=32,
    bottleneck_reduction=2,  # reduced bottleneck
)

# Your simulation data: [T, 3, H, W]
data = torch.randn(200,7, 3, 128, 128)  # replace with real data
print(data.shape)
data = torch.from_numpy(simsNP)
mask = torch.from_numpy(maskNP)
print(data.shape)

trained_model = train(model, data,mask, epochs=100, batch_size=8, lr=1e-3)

torch.Size([200, 7, 3, 128, 128])
torch.Size([20000, 7, 3, 128, 128])
Using device: cuda
epoch 1 | val loss 0.335975 | > (improvement of inf)
Epoch 0001 | Train Loss: 0.286817 | Val Loss:0.335975 > 0.274057, 0.307023, 0.312158 | LR: 1.00e-03
epoch 2 | val loss 0.323375 | > (improvement of 0.012600)
Epoch 0002 | Train Loss: 0.265719 | Val Loss:0.323375 > 0.260347, 0.310667, 0.319611 | LR: 1.00e-03
epoch 3 | val loss 0.315646 | > (improvement of 0.007729)
Epoch 0003 | Train Loss: 0.255681 | Val Loss:0.315646 > 0.253802, 0.305800, 0.312641 | LR: 1.00e-03
epoch 4 | val loss 0.310643 | > (improvement of 0.005003)
Epoch 0004 | Train Loss: 0.249412 | Val Loss:0.310643 > 0.248966, 0.304477, 0.312296 | LR: 1.00e-03
epoch 5 | val loss 0.304760 | > (improvement of 0.005883)
Epoch 0005 | Train Loss: 0.244317 | Val Loss:0.304760 > 0.243325, 0.302521, 0.311833 | LR: 1.00e-03
epoch 6 | val loss 0.301154 | > (improvement of 0.003606)
Epoch 0006 | Train Loss: 0.239445 | Val Loss:0.301154 > 0.239642, 0.